# Lesson 7A: Anomaly Detection - Theory

## Introduction

**Case Study**: Financial institutions like PayPal process billions of transactions yearly, where approximately 0.1% are fraudulent. The challenge is to identify these rare anomalous events among normal transactions.

**Anomaly detection** (also called outlier detection or novelty detection) identifies observations that deviate significantly from the majority of the data.

**Applications**:
- Financial fraud detection (credit cards, insurance claims)
- Network security (intrusion detection, DDoS attacks)
- Medical diagnosis (disease outbreak detection, patient monitoring)
- Industrial quality control (equipment failure prediction, defect detection)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.datasets import make_blobs

np.random.seed(42)
print("Libraries loaded successfully")

## Method 1: Isolation Forest

**Reference**: Liu, F. T., Ting, K. M., & Zhou, Z. H. (2008). "Isolation Forest". *IEEE International Conference on Data Mining*, 413-422.

### Theoretical Foundation

**Key Insight**: Anomalies are "few and different," making them more susceptible to isolation through random partitioning.

**Definition**: Given a dataset X, an isolation tree (iTree) recursively partitions the space by randomly selecting:
1. A feature dimension
2. A split value between the minimum and maximum values of that feature

**Anomaly Score**: For a point x, the anomaly score s(x, n) is defined as:

$$s(x, n) = 2^{-\frac{E(h(x))}{c(n)}}$$

where:
- h(x) is the path length from root to leaf for point x
- E(h(x)) is the average path length over an ensemble of iTrees
- c(n) is the average path length of unsuccessful search in Binary Search Tree
- c(n) = 2H(n-1) - 2(n-1)/n, where H(i) is the harmonic number

**Interpretation**:
- s → 1: Anomaly
- s → 0.5: Normal point
- s → 0: Safe region

### Algorithm

```
IsolationForest(X, t, ψ):
    Input: X - dataset, t - number of trees, ψ - subsample size
    Output: Anomaly scores for each point
    
    1. Initialize forest F = ∅
    2. Set height limit l = ceiling(log₂(ψ))
    3. For i = 1 to t:
        a. Sample X' ⊂ X with |X'| = ψ
        b. Build iTree(X', 0, l)
        c. Add tree to F
    4. For each x ∈ X:
        Compute s(x, ψ) using ensemble F
```

**Computational Complexity**:
- Training: O(t · ψ · log ψ)
- Testing: O(t · log ψ)

where t is number of trees, ψ is subsample size.

In [ ]:
# Isolation Forest - Simplified Implementation
class SimpleIsolationTree:
    """
    Simplified isolation tree for educational purposes.
    Production code should use sklearn.ensemble.IsolationForest
    """
    def __init__(self, max_depth=10):
        self.max_depth = max_depth
        self.split_feature = None
        self.split_value = None
    
    def fit(self, X, depth=0):
        """
        Recursively build isolation tree
        Returns: average path depth
        """
        n_samples, n_features = X.shape
        
        # Termination conditions
        if depth >= self.max_depth or n_samples <= 1:
            return depth
        
        # Random split selection
        self.split_feature = np.random.randint(n_features)
        feature_values = X[:, self.split_feature]
        self.split_value = np.random.uniform(feature_values.min(), feature_values.max())
        
        # Partition data
        left_mask = feature_values < self.split_value
        if left_mask.sum() == 0 or (~left_mask).sum() == 0:
            return depth
        
        # Recurse
        left_depth = self.fit(X[left_mask], depth + 1)
        right_depth = self.fit(X[~left_mask], depth + 1)
        
        return (left_depth + right_depth) / 2

# Demonstration
X_normal = np.random.randn(100, 2) * 0.5
X_anomaly = np.array([[4, 4], [-4, -4]])
X = np.vstack([X_normal, X_anomaly])

# Build ensemble
n_trees = 10
avg_depths = []
for i in range(len(X)):
    depths = []
    for _ in range(n_trees):
        tree = SimpleIsolationTree(max_depth=8)
        depth = tree.fit(X, 0)
        depths.append(depth)
    avg_depths.append(np.mean(depths))

avg_depths = np.array(avg_depths)
anomaly_scores = 1 / (avg_depths + 1)

print(f"Normal point average depth: {avg_depths[0]:.2f}")
print(f"Anomaly point average depth: {avg_depths[-1]:.2f}")
print(f"Conclusion: Anomalies have shorter paths (easier to isolate)")

## Method 2: Local Outlier Factor (LOF)

**Reference**: Breunig, M. M., Kriegel, H. P., Ng, R. T., & Sander, J. (2000). "LOF: Identifying Density-Based Local Outliers". *ACM SIGMOD International Conference on Management of Data*, 93-104.

### Theoretical Foundation

**Key Insight**: Anomalies exist in regions of lower density compared to their local neighborhoods.

**Mathematical Formulation**:

For a point p and integer k:

1. **k-distance(p)**: Distance to the k-th nearest neighbor

2. **Reachability Distance**:
$$\text{reach-dist}_k(p, o) = \max\{k\text{-distance}(o), d(p,o)\}$$

3. **Local Reachability Density (LRD)**:
$$\text{LRD}_k(p) = \frac{1}{\frac{\sum_{o \in N_k(p)} \text{reach-dist}_k(p, o)}{|N_k(p)|}}$$

where N_k(p) is the set of k-nearest neighbors of p

4. **Local Outlier Factor**:
$$\text{LOF}_k(p) = \frac{\sum_{o \in N_k(p)} \frac{\text{LRD}_k(o)}{\text{LRD}_k(p)}}{|N_k(p)|}$$

**Interpretation**:
- LOF ≈ 1: Point has similar density to neighbors (normal)
- LOF >> 1: Point is in lower density region (outlier)
- LOF << 1: Point is in higher density region (inlier)

**Computational Complexity**: O(n²) for naive implementation, O(n log n) with spatial indexing

In [ ]:
# LOF Demonstration
from sklearn.neighbors import NearestNeighbors

# Generate data with density variation
X_dense = np.random.randn(200, 2) * 0.3 + [0, 0]
X_sparse = np.random.randn(50, 2) * 0.3 + [3, 3]
X_outlier = np.array([[3, 0]])
X = np.vstack([X_dense, X_sparse, X_outlier])

# Compute k-nearest neighbors
k = 10
nbrs = NearestNeighbors(n_neighbors=k+1).fit(X)
distances, indices = nbrs.kneighbors(X)

# Visualize
plt.figure(figsize=(10, 6))
plt.scatter(X_dense[:, 0], X_dense[:, 1], c='blue', s=30, alpha=0.6, label='Dense cluster')
plt.scatter(X_sparse[:, 0], X_sparse[:, 1], c='green', s=30, alpha=0.6, label='Sparse cluster')
plt.scatter(X_outlier[:, 0], X_outlier[:, 1], c='red', s=200, marker='*', 
           edgecolors='black', linewidths=2, label='Outlier')

# Show k-NN of outlier
outlier_idx = len(X) - 1
neighbor_indices = indices[outlier_idx, 1:]
plt.scatter(X[neighbor_indices, 0], X[neighbor_indices, 1], 
           facecolors='none', edgecolors='red', s=100, linewidths=2)

plt.title('LOF: Outlier has Distant k-Nearest Neighbors', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Compute LOF scores
lof = LocalOutlierFactor(n_neighbors=k)
lof_scores = lof.fit_predict(X)
negative_factors = lof.negative_outlier_factor_

print(f"Dense cluster point LOF: {-negative_factors[0]:.2f}")
print(f"Sparse cluster point LOF: {-negative_factors[200]:.2f}")
print(f"Outlier LOF: {-negative_factors[-1]:.2f}")
print("Higher LOF indicates stronger outlier")

## Method 3: One-Class SVM

**Reference**: Schölkopf, B., Platt, J. C., Shawe-Taylor, J., Smola, A. J., & Williamson, R. C. (2001). "Estimating the Support of a High-Dimensional Distribution". *Neural Computation*, 13(7), 1443-1471.

### Theoretical Foundation

**Objective**: Find a hyperplane in feature space that separates the data from the origin with maximum margin.

**Optimization Problem**:

$$\min_{w, \rho, \xi} \frac{1}{2}\|w\|^2 - \rho + \frac{1}{\nu n}\sum_{i=1}^n \xi_i$$

Subject to:
$$w^T\phi(x_i) \geq \rho - \xi_i, \quad \xi_i \geq 0$$

where:
- φ(x) is the feature map (kernel function)
- ρ is the offset from origin
- ξᵢ are slack variables
- ν ∈ (0,1] controls the fraction of outliers and support vectors

**Decision Function**:
$$f(x) = \text{sign}(w^T\phi(x) - \rho)$$

**Kernel Trick**: Using kernel K(x, x') = φ(x)ᵀφ(x'), common choices:
- Linear: K(x, x') = xᵀx'
- RBF: K(x, x') = exp(-γ||x - x'||²)
- Polynomial: K(x, x') = (γxᵀx' + r)ᵈ

**Interpretation of ν**:
- Upper bound on fraction of outliers
- Lower bound on fraction of support vectors

In [ ]:
# Compare all three methods
X, _ = make_blobs(n_samples=300, centers=1, cluster_std=0.5, random_state=42)
X_anomalies = np.random.uniform(-5, 5, (20, 2))
X = np.vstack([X, X_anomalies])

models = {
    'Isolation Forest': IsolationForest(contamination=0.1, random_state=42),
    'LOF': LocalOutlierFactor(contamination=0.1),
    'One-Class SVM': OneClassSVM(nu=0.1, gamma='auto')
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, model) in zip(axes, models.items()):
    if name == 'LOF':
        labels = model.fit_predict(X)
    else:
        labels = model.fit(X).predict(X)
    
    # Plot results
    normal = labels == 1
    anomaly = labels == -1
    
    ax.scatter(X[normal, 0], X[normal, 1], c='blue', s=30, alpha=0.6, label='Normal')
    ax.scatter(X[anomaly, 0], X[anomaly, 1], c='red', s=80, marker='x', 
              linewidths=2, label='Anomaly')
    ax.set_title(name, fontweight='bold', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Anomaly Detection Methods Comparison', fontweight='bold', fontsize=16)
plt.tight_layout()
plt.show()

print("Detected anomalies:")
for name, model in models.items():
    if name == 'LOF':
        labels = model.fit_predict(X)
    else:
        labels = model.fit(X).predict(X)
    print(f"  {name}: {sum(labels == -1)} anomalies")

## Method Comparison and Selection

| Method | Best For | Computational Complexity | Hyperparameters |
|--------|----------|-------------------------|-----------------|
| **Isolation Forest** | High-dimensional data, global anomalies | O(t·ψ·log ψ) | t (trees), ψ (subsample) |
| **LOF** | Local anomalies, varying densities | O(n²) or O(n log n) | k (neighbors) |
| **One-Class SVM** | Complex boundaries, kernel methods | O(n²) to O(n³) | ν (outlier fraction), kernel |

### Selection Guidelines

**Use Isolation Forest when**:
- High-dimensional data (d > 50)
- Large datasets (n > 10,000)
- Need for computational efficiency
- Global anomaly detection sufficient

**Use LOF when**:
- Local density variations important
- Small to medium datasets (n < 10,000)
- Willing to sacrifice computation for accuracy
- Need to detect clustered anomalies

**Use One-Class SVM when**:
- Complex, non-linear decision boundaries
- Kernel methods applicable
- Small datasets with high precision requirements
- Domain knowledge suggests specific kernel

### Production Considerations

1. **Scalability**: Isolation Forest scales best to large datasets
2. **Interpretability**: Isolation Forest provides path length interpretation
3. **Parameter Sensitivity**: LOF requires careful k selection
4. **Online Learning**: Consider streaming variants for real-time applications

## References

1. Liu, F. T., Ting, K. M., & Zhou, Z. H. (2008). "Isolation Forest". *IEEE International Conference on Data Mining (ICDM)*, 413-422.

2. Breunig, M. M., Kriegel, H. P., Ng, R. T., & Sander, J. (2000). "LOF: Identifying Density-Based Local Outliers". *ACM SIGMOD International Conference on Management of Data*, 93-104.

3. Schölkopf, B., Platt, J. C., Shawe-Taylor, J., Smola, A. J., & Williamson, R. C. (2001). "Estimating the Support of a High-Dimensional Distribution". *Neural Computation*, 13(7), 1443-1471.

4. Chandola, V., Banerjee, A., & Kumar, V. (2009). "Anomaly Detection: A Survey". *ACM Computing Surveys*, 41(3), 1-58.

5. Aggarwal, C. C. (2017). *Outlier Analysis* (2nd ed.). Springer.

## Further Reading

- Rousseeuw, P. J., & Hubert, M. (2011). "Robust statistics for outlier detection". *Wiley Interdisciplinary Reviews: Data Mining and Knowledge Discovery*, 1(1), 73-79.

- Goldstein, M., & Uchida, S. (2016). "A Comparative Evaluation of Unsupervised Anomaly Detection Algorithms for Multivariate Data". *PLoS ONE*, 11(4).

- Pang, G., Shen, C., Cao, L., & Hengel, A. V. D. (2021). "Deep Learning for Anomaly Detection: A Review". *ACM Computing Surveys*, 54(2), 1-38.